In [2]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

# 1. โหลดข้อมูล
train_df = pd.read_csv("../cleaned_data/train_data.csv")
test_df = pd.read_csv("../cleaned_data/test_data.csv")

# 2. Feature Engineering
def engineer_features(df):
    df = df.copy()
    df['is_senior'] = (df['age'] >= 60).astype(int)
    df['has_free_cash'] = ((df['balance'] > 0) & (df['housing'] == 'no') & (df['loan'] == 'no')).astype(int)
    df['total_contacts'] = df['campaign'] + df['previous']
    df['is_balance_negative'] = (df['balance'] < 0).astype(int)
    return df

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

X_train = train_fe.drop(columns=['y'])
y_train = train_fe['y']
X_test = test_fe.drop(columns=['y'])
y_test = test_fe['y']

# 3. One-Hot Encoding
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# 4. Feature Scaling (XGBoost รับข้อมูลทั้งหมดได้โดยไม่ต้องตัดฟีเจอร์ เพราะมีระบบ Regularization ในตัว)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_encoded), columns=X_train_encoded.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_encoded), columns=X_test_encoded.columns)

# 5. คำนวณน้ำหนักสำหรับคลาส 1 (สัดส่วนจำนวน 0 ต่อ 1)
ratio = float((y_train == 0).sum()) / (y_train == 1).sum()

# 6. Hyperparameter Tuning
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(
    random_state=0, 
    scale_pos_weight=ratio, # จัดการ Imbalanced Data
    eval_metric='logloss'
)

random_search = RandomizedSearchCV(
    estimator=xgb_base, 
    param_distributions=param_dist, 
    n_iter=15, 
    cv=5, 
    scoring='f1', 
    random_state=0, 
    n_jobs=-1
)

print("กำลังเทรน XGBoost...")
random_search.fit(X_train_scaled, y_train)
best_xgb = random_search.best_estimator_

# 7. ประเมินผล
y_probs = best_xgb.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_probs >= 0.50).astype(int)

print("\n=== XGBOOST MODEL REPORT ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(best_xgb, "../models/xgb_model.pkl")
print("✅ บันทึก XGBoost เรียบร้อย!")

กำลังเทรน XGBoost...

=== XGBOOST MODEL REPORT ===
Accuracy: 0.8352818613265425

Confusion Matrix:
 [[6631 1004]
 [ 419  585]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.87      0.90      7635
           1       0.37      0.58      0.45      1004

    accuracy                           0.84      8639
   macro avg       0.65      0.73      0.68      8639
weighted avg       0.87      0.84      0.85      8639

✅ บันทึก XGBoost เรียบร้อย!
